# Experiments - SKILL 2025

In [271]:
import os
import pandas as pd
from config import algorithm_strategy_pairs
from utils.simulation_utils import get_directory_for_algorithm
from scipy.stats import chi2

In [272]:
from IPython.display import display

def get_sorted_results_by_regret(arms, timestep=1000000, dof=999999, null_variance=None):
    """
    Combine average results, variance calculations, suboptimal arm ratio, and p-value for given arms and timestep.

    Returns:
        pandas.DataFrame: Combined DataFrame with average, variance, suboptimal ratio, and p-value columns.
    """
    avg_results = {}
    var_results = {}
    suboptimal_ratios = {}

    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        avg_path = os.path.join(results_dir, f'average_results_{combination_name}.csv')
        var_path = os.path.join(results_dir, f'results_{combination_name}.csv')
        strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
        results_key = f"{algorithm.name}_{strategy_suffix}"

        # Average results
        if os.path.exists(avg_path):
            df_avg = pd.read_csv(avg_path)
            filtered_avg = df_avg[df_avg['Timestep'] == timestep]
            if not filtered_avg.empty:
                avg_results[results_key] = filtered_avg.iloc[0]
                # Suboptimal arm ratio
                suboptimal_ratio = filtered_avg['Average Suboptimal Arms'].values[0] / timestep
                suboptimal_ratios[results_key] = suboptimal_ratio
            else:
                suboptimal_ratios[results_key] = None
        else:
            suboptimal_ratios[results_key] = None
        
        # Variance results
        if os.path.exists(var_path):
            df_var = pd.read_csv(var_path)
            filtered_var = df_var[df_var['Timestep'] == timestep]
            if not filtered_var.empty:
                reward_variance = filtered_var['Total Reward'].var()
                avg_reward = filtered_var['Total Reward'].mean()
                avg_regret = filtered_var['Total Regret'].mean()
            else:
                reward_variance = None
                avg_reward = None
                avg_regret = None
            var_results[results_key] = {
                "Reward Variance": reward_variance,
                #"Average Reward": avg_reward,
                "Average Regret": avg_regret
            }
        else:
            var_results[results_key] = {
                "Reward Variance": None,
                #"Average Reward": None,
                "Average Regret": None
            }

    # Merge results
    avg_df = pd.DataFrame.from_dict(avg_results, orient='index')
    var_df = pd.DataFrame.from_dict(var_results, orient='index')
    combined_df = avg_df.join(var_df, rsuffix='_var')

    # Add suboptimal ratio column
    subopt_df = pd.DataFrame.from_dict(suboptimal_ratios, orient='index', columns=['Suboptimal Arm Ratio'])
    combined_df = combined_df.join(subopt_df)

    # Remove duplicate columns (those with _var suffix)
    duplicate_cols = [col for col in combined_df.columns if col.endswith('_var')]
    combined_df = combined_df.drop(columns=duplicate_cols)
    combined_df = combined_df.drop(columns=['Timestep'])

    # Remove 'Average Reward' column if present (since it's the same as 'Average Total Reward')
    if "Average Reward" in combined_df.columns:
        combined_df = combined_df.drop(columns=["Average Reward"])
    
    if "Average Total Reward" in combined_df.columns:
        combined_df = combined_df.drop(columns=["Average Total Reward"])

    # Calculate p-value for variance and add as a column
    def calc_p_value(row):
        observed_var = row.get('Reward Variance', None)
        if observed_var is not None:
            chi2_stat = dof * observed_var / null_variance
            return 1 - chi2.cdf(chi2_stat, dof)
        return None

    combined_df['Variance p-value'] = combined_df.apply(calc_p_value, axis=1)

    # Optional: sort by Average Regret if present
    if "Average Regret" in combined_df.columns:
        combined_df = combined_df.sort_values(by="Average Regret", ascending=False)

    # Round all columns except 'Suboptimal Arm Ratio' to two decimals
    for col in combined_df.columns:
        if col != 'Suboptimal Arm Ratio':
            combined_df[col] = combined_df[col].apply(lambda x: round(x, 2) if pd.notnull(x) else x)
        if col == 'Suboptimal Arm Ratio':
            combined_df[col] = combined_df[col].apply(lambda x: round(x, 5) if pd.notnull(x) else x)

    display(combined_df.style.format({
        'Reward Variance': '{:,.2f}',
        'Variance p-value': '{:.2e}',
        'Suboptimal Arm Ratio': '{:.6f}'
    }).set_caption("Algorithm Results at Timestep 1,000,000 and Variance Significance sorted by Average Regret"))

    # For saving the results to a CSV file, uncomment the following line:
    #combined_df.to_csv("sorted_results_by_regret.csv")

    return combined_df


## Scenario A: Baseline

In [273]:
sorted_results_by_regret = get_sorted_results_by_regret([0.8, 0.9], timestep=1000000, dof=999999, null_variance=90000)

,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Suboptimal Arm Ratio,Variance p-value
UCB-Improved_delta-1,325001.060000,32500.110000,132486.990000,867513.010000,"2,143,278,875.08",0.325000,0.00e+00
Greedy_epsilon-0.5,250022.030000,25002.200000,124979.360000,875020.640000,"99,490.43",0.250020,0.00e+00
ETC_exploration_rounds-100000,100000.000000,10000.000000,110002.400000,889997.600000,"83,438.69",0.100000,1.00e+00
ETC_exploration_rounds-10,70040.820000,7004.080000,106989.530000,893010.470000,"658,647,565.52",0.070040,0.00e+00
Greedy_epsilon-0.1,50084.760000,5008.480000,104998.690000,895001.310000,"80,048.78",0.050080,1.00e+00
Greedy_epsilon-0.05,25184.690000,2518.470000,102516.590000,897483.410000,"84,563.72",0.025180,1.00e+00
ETC_exploration_rounds-10000,10000.000000,1000.000000,100996.660000,899003.340000,"77,952.99",0.010000,1.00e+00
Greedy_epsilon-0.01,5880.140000,588.010000,100577.620000,899422.380000,"107,363.81",0.005880,0.00e+00
Greedy_epsilon-0.005,4229.880000,422.990000,100418.570000,899581.430000,"106,607.54",0.004230,0.00e+00
UCB_default,2381.750000,238.170000,100233.810000,899766.190000,"78,515.43",0.002380,1.00e+00


## Scenario B: Low-Variance Micro-Gap

In [274]:
sorted_results_by_regret = get_sorted_results_by_regret([0.895, 0.9], timestep=1000000, dof=999999, null_variance=90000)

,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Suboptimal Arm Ratio,Variance p-value
UCB-Improved_delta-1,505000.000000,2525.000000,102523.120000,897476.880000,"6,194,415.18",0.505000,0.00e+00
ETC_exploration_rounds-10,430160.030000,2150.800000,102151.030000,897848.970000,"6,432,726.05",0.430160,0.00e+00
ETC_exploration_rounds-100,410103.960000,2050.520000,102054.670000,897945.330000,"6,280,731.62",0.410100,0.00e+00
Greedy_epsilon-0.005,285740.620000,1428.700000,101425.390000,898574.610000,"3,665,457.69",0.285740,0.00e+00
ETC_exploration_rounds-1000,258225.420000,1291.130000,101287.740000,898712.260000,"4,837,381.73",0.258230,0.00e+00
Greedy_epsilon-0.5,252903.260000,1264.520000,101262.240000,898737.760000,"77,552.71",0.252900,1.00e+00
UCB_default,225434.510000,1127.170000,101119.980000,898880.020000,"91,872.83",0.225430,0.00e+00
Greedy_epsilon-0.01,220636.240000,1103.180000,101092.950000,898907.050000,"2,434,165.12",0.220640,0.00e+00
ETC_exploration_rounds-100000,100000.000000,500.000000,100491.100000,899508.900000,"79,356.56",0.100000,1.00e+00
EUCBV_rho-0.5,92272.120000,461.360000,100453.650000,899546.350000,"109,142.90",0.092270,0.00e+00


## Scenario C: High-Variance Micro-Gap

In [275]:
sorted_results_by_regret = get_sorted_results_by_regret([0.89, 0.895], timestep=1000000, dof=999999, null_variance=93975)

,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Suboptimal Arm Ratio,Variance p-value
UCB-Improved_delta-1,490000.140000,2450.000000,107456.680000,892543.320000,"6,169,245.59",0.490000,0.00e+00
ETC_exploration_rounds-10,429999.160000,2150.000000,107153.970000,892846.030000,"6,453,855.77",0.430000,0.00e+00
ETC_exploration_rounds-100,400123.020000,2000.620000,107008.360000,892991.640000,"6,217,677.79",0.400120,0.00e+00
Greedy_epsilon-0.005,352517.890000,1762.590000,106774.640000,893225.360000,"4,012,222.43",0.352520,0.00e+00
Greedy_epsilon-0.5,253809.300000,1269.050000,106277.850000,893722.150000,"82,824.39",0.253810,1.00e+00
Greedy_epsilon-0.01,240771.050000,1203.850000,106212.530000,893787.470000,"2,644,339.00",0.240770,0.00e+00
UCB_default,234514.580000,1172.570000,106180.440000,893819.560000,"93,873.80",0.234510,7.80e-01
ETC_exploration_rounds-1000,231292.990000,1156.460000,106168.500000,893831.500000,"4,469,433.28",0.231290,0.00e+00
ETC_exploration_rounds-100000,100000.000000,500.000000,105506.460000,894493.540000,"77,761.99",0.100000,1.00e+00
EUCBV_rho-0.5,95262.970000,476.310000,105485.880000,894514.120000,"96,233.97",0.095260,0.00e+00
